## Imports

In [2]:
import pandas as pd
import random
import os
import time
# import openai
# import anthropic
# import google.generativeai as genai


## Valdation

### MCP 30 Samples (Old Bucket Approach)

In [3]:
# Load your full MCP dataset
mcp_df = pd.read_csv("data/mcp_desc_all_nov_7_cleaned.csv")

# --- Define keywords (expanded sets) ---
bucket_1_keywords = [
    "search", "query", "retrieve", "scrape", "crawl", "research",
    "knowledge", "api", "data"
]
bucket_2_keywords = [
    "generate", "image", "video", "audio", "tts", "voice", "picture",
    "text", "story", "art", "music"
]
bucket_3_keywords = [
    "automation", "code", "execute", "browser", "workflow", "tool",
    "script", "process", "run", "system"
]

# Lowercase text for matching
mcp_df["text_lower"] = mcp_df["text_for_llm"].str.lower()

# Function to count keyword hits
def count_hits(text, keywords):
    return sum(1 for kw in keywords if kw in text)

# Assign each MCP to a bucket
buckets = []
for text in mcp_df["text_lower"]:
    counts = [
        count_hits(text, bucket_1_keywords),
        count_hits(text, bucket_2_keywords),
        count_hits(text, bucket_3_keywords)
    ]
    # Pick the bucket with the most hits
    max_hits = max(counts)
    if max_hits == 0:
        buckets.append("unclassified")
    else:
        # tie-break: default to automation bucket (index 2)
        bucket_idx = counts.index(max_hits) if counts.count(max_hits) == 1 else 2
        buckets.append(["retrieval", "generative", "automation"][bucket_idx])

mcp_df["bucket"] = buckets

# --- Sample 10 from each of the three buckets ---
samples = []
for label in ["retrieval", "generative", "automation"]:
    subset = mcp_df[mcp_df["bucket"] == label]
    if len(subset) >= 10:
        samples.append(subset.sample(n=10, random_state=41))
    else:
        samples.append(subset)  # if fewer than 10 exist

sampled_df = pd.concat(samples)

# --- Save final 30-row dataset ---
sampled_df = sampled_df[
    ["title", "url", "uploaded_clean", "text_for_llm", "len_text", "bucket"]
]
sampled_df = sampled_df.drop(columns=["uploaded_clean", "len_text"])
# sampled_df.to_csv("data/mcp_30_sample.csv", index=False)

print(mcp_df["bucket"].value_counts().sort_index())

bucket
automation      5594
generative       196
retrieval        995
unclassified       1
Name: count, dtype: int64


### 23 Additional Samples

In [4]:
# Load data
mcp_df = pd.read_csv("data/mcp_desc_all_nov_7_cleaned.csv")

# Get 23 more random samples
additional_samples = mcp_df.sample(n=23, random_state=42)

# Save
additional_samples[["title", "url", "text_for_llm"]].to_csv("data/mcp_23_additional.csv", index=False)

### O*NET Data

In [20]:
# Load the O*NET CSV
onet_df = pd.read_csv("../onet_hierarchy/onet_tasks_dwa_iwa_gwa.csv")

# Keep only the columns relevant for the cascading sheet
onet_hierarchy = onet_df[[
    "gwa_title", "iwa_title", "dwa_title", "task"
]].drop_duplicates().reset_index(drop=True)

# Save for Google Sheets use
onet_hierarchy.to_csv("data/onet_options_sheet.csv", index=False)
